### OD DATA PREPARATION

* Raw data from SRT

In [37]:
import re
import zipfile
import datetime
import tempfile
import io
from pathlib import Path

import openpyxl
import pandas as pd
import rarfile

In [22]:
# CONFIGURATION
DATA_ROOT = Path('DATA')
OUTPUT_DIR = Path('OUTPUT')
OUTPUT_DIR.mkdir(exist_ok=True)

STATIONS = ['TLC','BMR','BSN','KTW','CTK','WSN','BKH','TSH','LAK','KHA','DMG','LHK','RST']

ARCHIVE_PATTERN = re.compile(r'PassengerOD_(\d{8})\.(zip|rar)$', re.IGNORECASE)
FILE_PATTERN = re.compile(r'PassengerOD_(\d{8})_(\d{2})-(\d{2})\.xlsx$')
rarfile.UNRAR_TOOL = r"E:\Programs\WinRAR\UnRAR.exe"

In [23]:
# FUNCTION: DATE CONVERSION
def be_to_ce_date(ddmmyyyy_be):
    dd = int(ddmmyyyy_be[0:2])
    mm = int(ddmmyyyy_be[2:4])
    yyyy_be = int(ddmmyyyy_be[4:8])
    return datetime.date(yyyy_be - 543, mm, dd)

In [24]:
# FUNCTION: ARCHIVE EXTRACTION
def extract_archive(archive_path, dest_dir):
    suffix = archive_path.suffix.lower()
    if suffix == '.zip':
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
    elif suffix == '.rar':
        with rarfile.RarFile(archive_path) as rf:
            rf.extractall(dest_dir)
    else:
        raise ValueError(f"Unsupported archive type: {archive_path}")

In [25]:
# FUNCTION: PARSE ONE HOURLY XLSX
def parse_od_file(filepath, service_date_be):
    fname = Path(filepath).name
    m = FILE_PATTERN.match(fname)
    if not m:
        raise ValueError(f"Filename doesn't match expected pattern: {fname}")
    _, hour_start, hour_end = m.groups()
    hour_start, hour_end = int(hour_start), int(hour_end)

    wb = openpyxl.load_workbook(filepath, data_only=True)
    ws = wb['data']
    rows = list(ws.iter_rows(values_only=True))

    header = rows[6]
    col_stations = header[1:14]
    assert list(col_stations) == STATIONS, f"Column header mismatch in {fname}: {col_stations}"

    records = []
    for r in rows[7:20]:
        origin = r[0]
        if origin == 'Total Exit':
            break
        for j, dest in enumerate(STATIONS):
            count = r[1 + j]
            records.append({
                'service_date': service_date_be,
                'hour_start': hour_start,
                'hour_end': hour_end,
                'origin': origin,
                'destination': dest,
                'count': float(count) if count is not None else 0.0,
            })
    df = pd.DataFrame.from_records(records)

    entry_rows = [r for r in rows[7:21] if r[0] in STATIONS]
    reported_entry = {r[0]: r[14] for r in entry_rows}
    exit_row = [r for r in rows[7:21] if r[0] == 'Total Exit'][0]
    reported_exit = dict(zip(STATIONS, exit_row[1:14]))

    computed_entry = df.groupby('origin')['count'].sum().to_dict()
    computed_exit = df.groupby('destination')['count'].sum().to_dict()

    mismatches = []
    for s in STATIONS:
        ce, re_ = computed_entry.get(s, 0), (reported_entry.get(s) or 0)
        cx, rx = computed_exit.get(s, 0), (reported_exit.get(s) or 0)
        if abs(ce - re_) > 1e-6:
            mismatches.append(('entry', s, ce, re_))
        if abs(cx - rx) > 1e-6:
            mismatches.append(('exit', s, cx, rx))

    return df, mismatches

In [26]:
# FUNCTION: PARSE ONE DAY'S EXTRACTED FOLDER
def parse_day_dir(day_dir, service_date_be):
    day_dir = Path(day_dir)
    files = sorted(day_dir.glob('PassengerOD_*.xlsx'))
    expected_hours = set(range(5, 24)) | {0}

    all_dfs, all_mismatches, found_hours = [], [], set()
    for f in files:
        df, mism = parse_od_file(f, service_date_be)
        all_dfs.append(df)
        all_mismatches.extend([(f.name, *m) for m in mism])
        found_hours.add(df['hour_start'].iloc[0])

    missing_hours = sorted(expected_hours - found_hours)
    if missing_hours:
        print(f"  WARNING [{service_date_be}]: missing hours {missing_hours}")

    if not all_dfs:
        return pd.DataFrame(), all_mismatches

    combined = pd.concat(all_dfs, ignore_index=True)
    combined['service_date_ce'] = be_to_ce_date(service_date_be)
    return combined, all_mismatches

In [27]:
# FUNCTION: PROCESS ONE MONTH
def process_month(month_dir):
    month_dir = Path(month_dir)
    archives = sorted(p for p in month_dir.iterdir() if ARCHIVE_PATTERN.match(p.name))

    all_dfs = []
    all_mismatches = []
    for archive in archives:
        m = ARCHIVE_PATTERN.match(archive.name)
        service_date_be = m.group(1)
        print(f"Processing {archive.name} ...")

        with tempfile.TemporaryDirectory() as tmpdir:
            try:
                extract_archive(archive, tmpdir)
            except Exception as e:
                print(f"  ERROR extracting {archive.name}: {e}")
                continue
            df, mism = parse_day_dir(tmpdir, service_date_be)
            if df.empty:
                print(f"  WARNING: no data parsed for {archive.name}")
                continue
            all_dfs.append(df)
            all_mismatches.extend([(archive.name, *m) for m in mism])

    if not all_dfs:
        return pd.DataFrame(), all_mismatches

    return pd.concat(all_dfs, ignore_index=True), all_mismatches

In [10]:
# ONE MONTH TEST RUN
df_202506, mismatches_202506 = process_month(DATA_ROOT / '202506')
print(f'SHAPE: {df_202506.shape}')
print(f'MISMATCHES: {len(mismatches_202506)}')
df_202506.head()

Processing PassengerOD_01062568.zip ...


C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing PassengerOD_02062568.zip ...
Processing PassengerOD_03062568.zip ...
Processing PassengerOD_04062568.zip ...
Processing PassengerOD_05062568.zip ...
Processing PassengerOD_06062568.zip ...
Processing PassengerOD_07062568.zip ...
Processing PassengerOD_08062568.zip ...
Processing PassengerOD_09062568.zip ...
Processing PassengerOD_10062568.zip ...
Processing PassengerOD_11062568.zip ...
Processing PassengerOD_12062568.zip ...
Processing PassengerOD_13062568.zip ...
Processing PassengerOD_14062568.zip ...
Processing PassengerOD_15062568.zip ...
Processing PassengerOD_16062568.zip ...
Processing PassengerOD_17062568.zip ...
Processing PassengerOD_18062568.zip ...
Processing PassengerOD_19062568.zip ...
Processing PassengerOD_20062568.zip ...
Processing PassengerOD_21062568.zip ...
Processing PassengerOD_22062568.zip ...
Processing PassengerOD_23062568.zip ...
Processing PassengerOD_24062568.zip ...
Processing PassengerOD_25062568.zip ...
Processing PassengerOD_26062568.zip ...


,service_date,hour_start,hour_end,origin,destination,count,service_date_ce
0,01062568,5,6,TLC,TLC,0.0,2025-06-01
1,01062568,5,6,TLC,BMR,0.0,2025-06-01
2,01062568,5,6,TLC,BSN,0.0,2025-06-01
3,01062568,5,6,TLC,KTW,0.0,2025-06-01
4,01062568,5,6,TLC,CTK,0.0,2025-06-01


In [ ]:
# EXPORT TO CSV (SINGLE MONTH TEST)
df_202506 = df_202506.drop(columns=['service_date'])
df_202506.to_csv(OUTPUT_DIR / 'od_202506.csv', index=False)

In [28]:
# EXPORT TO CSV (12 MONTHS LOOP)
month_dirs = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir())
for month_dir in month_dirs:
    print(f"\n=== {month_dir.name} ===")
    df, mism = process_month(month_dir)
    if df.empty:
        continue
    df = df.drop(columns=['service_date'])
    df.to_csv(OUTPUT_DIR / f"od_{month_dir.name}.csv", index=False)
    if mism:
        pd.DataFrame(mism, columns=['file','direction','station','computed','reported']).to_csv(
            OUTPUT_DIR / f"mismatches_{month_dir.name}.csv", index=False)
    print(f"  Saved {len(df):,} rows; {len(mism)} mismatches")


=== 202506 ===
Processing PassengerOD_01062568.zip ...


C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing PassengerOD_02062568.zip ...
Processing PassengerOD_03062568.zip ...
Processing PassengerOD_04062568.zip ...
Processing PassengerOD_05062568.zip ...
Processing PassengerOD_06062568.zip ...
Processing PassengerOD_07062568.zip ...
Processing PassengerOD_08062568.zip ...
Processing PassengerOD_09062568.zip ...
Processing PassengerOD_10062568.zip ...
Processing PassengerOD_11062568.zip ...
Processing PassengerOD_12062568.zip ...
Processing PassengerOD_13062568.zip ...
Processing PassengerOD_14062568.zip ...
Processing PassengerOD_15062568.zip ...
Processing PassengerOD_16062568.zip ...
Processing PassengerOD_17062568.zip ...
Processing PassengerOD_18062568.zip ...
Processing PassengerOD_19062568.zip ...
Processing PassengerOD_20062568.zip ...
Processing PassengerOD_21062568.zip ...
Processing PassengerOD_22062568.zip ...
Processing PassengerOD_23062568.zip ...
Processing PassengerOD_24062568.zip ...
Processing PassengerOD_25062568.zip ...
Processing PassengerOD_26062568.zip ...


C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing PassengerOD_02072568.zip ...
Processing PassengerOD_03072568.zip ...
Processing PassengerOD_04072568.zip ...
Processing PassengerOD_05072568.zip ...
Processing PassengerOD_06072568.zip ...
Processing PassengerOD_07072568.zip ...
Processing PassengerOD_08072568.zip ...
Processing PassengerOD_09072568.zip ...
Processing PassengerOD_10072568.zip ...
Processing PassengerOD_11072568.zip ...
Processing PassengerOD_12072568.zip ...
Processing PassengerOD_13072568.zip ...
Processing PassengerOD_14072568.zip ...
Processing PassengerOD_15072568.zip ...
Processing PassengerOD_16072568.zip ...
Processing PassengerOD_17072568.zip ...
Processing PassengerOD_18072568.zip ...
Processing PassengerOD_19072568.zip ...
Processing PassengerOD_20072568.zip ...
Processing PassengerOD_21072568.zip ...
Processing PassengerOD_22072568.zip ...
Processing PassengerOD_23072568.rar ...
Processing PassengerOD_24072568.rar ...
Processing PassengerOD_25072568.zip ...
Processing PassengerOD_26072568.zip ...


...

Processing PassengerOD_02032569.zip ...
Processing PassengerOD_03032569.zip ...
Processing PassengerOD_04032569.zip ...
Processing PassengerOD_05032569.zip ...
Processing PassengerOD_06032569.zip ...
Processing PassengerOD_07032569.zip ...
Processing PassengerOD_08032569.zip ...
Processing PassengerOD_09032569.zip ...
Processing PassengerOD_10032569.zip ...
Processing PassengerOD_11032569.zip ...
Processing PassengerOD_12032569.zip ...
Processing PassengerOD_13032569.zip ...
Processing PassengerOD_14032569.zip ...
Processing PassengerOD_15032569.zip ...
Processing PassengerOD_16032569.zip ...
Processing PassengerOD_17032569.zip ...
Processing PassengerOD_18032569.zip ...
Processing PassengerOD_19032569.zip ...
Processing PassengerOD_20032569.zip ...
Processing PassengerOD_21032569.zip ...
Processing PassengerOD_22032569.zip ...
Processing PassengerOD_23032569.zip ...
Processing PassengerOD_24032569.zip ...
Processing PassengerOD_25032569.zip ...
Processing PassengerOD_26032569.zip ...


C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing PassengerOD_02042569.zip ...
Processing PassengerOD_03042569.zip ...
Processing PassengerOD_04042569.zip ...
Processing PassengerOD_05042569.zip ...
Processing PassengerOD_06042569.zip ...
Processing PassengerOD_07042569.zip ...
Processing PassengerOD_08042569.zip ...
Processing PassengerOD_09042569.zip ...
Processing PassengerOD_10042569.zip ...
Processing PassengerOD_11042569.zip ...
Processing PassengerOD_12042569.zip ...
Processing PassengerOD_13042569.zip ...
Processing PassengerOD_14042569.zip ...
Processing PassengerOD_15042569.zip ...
Processing PassengerOD_16042569.zip ...
Processing PassengerOD_17042569.zip ...
Processing PassengerOD_18042569.zip ...
Processing PassengerOD_19042569.zip ...
Processing PassengerOD_20042569.zip ...
Processing PassengerOD_21042569.zip ...
Processing PassengerOD_22042569.zip ...
Processing PassengerOD_23042569.zip ...
Processing PassengerOD_24042569.zip ...
Processing PassengerOD_25042569.zip ...
Processing PassengerOD_26042569.zip ...


C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Processing PassengerOD_02052569.zip ...
Processing PassengerOD_03052569.zip ...
Processing PassengerOD_04052569.zip ...
Processing PassengerOD_05052569.zip ...
Processing PassengerOD_06052569.zip ...
Processing PassengerOD_07052569.zip ...
Processing PassengerOD_08052569.zip ...
Processing PassengerOD_09052569.zip ...
Processing PassengerOD_10052569.zip ...
Processing PassengerOD_11052569.zip ...
Processing PassengerOD_12052569.zip ...
Processing PassengerOD_13052569.zip ...
Processing PassengerOD_14052569.zip ...
Processing PassengerOD_15052569.zip ...
Processing PassengerOD_16052569.zip ...
Processing PassengerOD_17052569.zip ...
Processing PassengerOD_18052569.zip ...
Processing PassengerOD_19052569.zip ...
Processing PassengerOD_20052569.zip ...
Processing PassengerOD_21052569.zip ...
Processing PassengerOD_22052569.zip ...
Processing PassengerOD_23052569.zip ...
Processing PassengerOD_24052569.zip ...
Processing PassengerOD_25052569.zip ...
Processing PassengerOD_26052569.zip ...


In [ ]:
# CHECK FOR DUPLICATE IN OCTOBER 1
df10 = pd.read_csv('OUTPUT/od_202510.csv')
counts = df10.groupby('service_date_ce')['hour_start'].nunique()
print(counts[counts != 20])

Series([], Name: hour_start, dtype: int64)


In [ ]:
# CHECK FOR DUPLICATE IN OCTOBER 2
dup10 = df10[df10.duplicated(subset=['service_date_ce','hour_start','origin','destination'], keep=False)]
print(dup10['service_date_ce'].unique())
print(dup10.groupby(['service_date_ce','hour_start']).size())

<StringArray>
['2025-10-16']
Length: 1, dtype: str
service_date_ce  hour_start
2025-10-16       20            338
dtype: int64


In [ ]:
# CHECK FOR DUPLICATE IN OCTOBER 3
with zipfile.ZipFile('DATA/202510/PassengerOD_16102568.zip') as zf:
    print(zf.namelist())

['PassengerOD_16102568_05-06.xlsx', 'PassengerOD_16102568_06-07.xlsx', 'PassengerOD_16102568_07-08.xlsx', 'PassengerOD_16102568_08-09.xlsx', 'PassengerOD_16102568_09-10.xlsx', 'PassengerOD_16102568_10-11.xlsx', 'PassengerOD_16102568_11-12.xlsx', 'PassengerOD_16102568_12-13.xlsx', 'PassengerOD_16102568_13-14.xlsx', 'PassengerOD_16102568_14-15.xlsx', 'PassengerOD_16102568_15-16.xlsx', 'PassengerOD_16102568_16-17.xlsx', 'PassengerOD_16102568_17-18.xlsx', 'PassengerOD_16102568_18-19.xlsx', 'PassengerOD_16102568_19-20.xlsx', 'PassengerOD_16102568_20-20.xlsx', 'PassengerOD_16102568_20-21.xlsx', 'PassengerOD_16102568_21-22.xlsx', 'PassengerOD_16102568_22-23.xlsx', 'PassengerOD_16102568_23-00.xlsx', 'PassengerOD_17102568_00-01.xlsx']


In [ ]:
# CHECK FOR DUPLICATE IN OCTOBER 4
with zipfile.ZipFile('DATA/202510/PassengerOD_16102568.zip') as zf:
    for fname in ['PassengerOD_16102568_20-20.xlsx', 'PassengerOD_16102568_20-21.xlsx']:
        with zf.open(fname) as f:
            wb = openpyxl.load_workbook(io.BytesIO(f.read()), data_only=True)
            ws = wb['data']
            rows = list(ws.iter_rows(values_only=True))
            print(fname)
            print(rows[3])  # business day header row
            print(rows[6])  # column header
            print(rows[7])  # first data row
            print()

PassengerOD_16102568_20-20.xlsx
(None, None, None, 'Start Business Day : 16/10/2568 20:00:00', None, None, None, None, 'TO End Business Day : 16/10/2568 20:00:00', None, None, None, None, None, None)
('Station', 'TLC', 'BMR', 'BSN', 'KTW', 'CTK', 'WSN', 'BKH', 'TSH', 'LAK', 'KHA', 'DMG', 'LHK', 'RST', 'Total Entry')
('TLC', 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)

PassengerOD_16102568_20-21.xlsx
(None, None, None, 'Start Business Day : 16/10/2568 20:00:00', None, None, None, None, 'TO End Business Day : 16/10/2568 21:00:00', None, None, None, None, None, None)
('Station', 'TLC', 'BMR', 'BSN', 'KTW', 'CTK', 'WSN', 'BKH', 'TSH', 'LAK', 'KHA', 'DMG', 'LHK', 'RST', 'Total Entry')
('TLC', 0.0, 1.0, 2.0, 4.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 2.0, 3.0, 10.0, 25.0)



C:\Users\likre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
# DROP DUPLICATE
# 20-20.xlsx is a degenerate/erroneous report (zero-duration window 20:00→20:00, all-zero data)
df10 = df10[df10['hour_start'] != df10['hour_end']]
print(df10.shape)
df10.to_csv('OUTPUT/od_202510.csv', index=False)

(104780, 6)


In [39]:
# CHECK FOR DUPLICATE IN DECEMBER 1
df12 = pd.read_csv('OUTPUT/od_202512.csv')
hrs = df12[df12['service_date_ce']=='2025-12-31']['hour_start'].unique()
print(sorted(hrs))
print(df12[df12['service_date_ce']=='2025-12-31'].groupby('hour_start').size())

[np.int64(0), np.int64(1), np.int64(2), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]
hour_start
0     169
1     169
2     169
5     169
6     169
7     169
8     169
9     169
10    169
11    169
12    169
13    169
14    169
15    169
16    169
17    169
18    169
19    169
20    169
21    169
22    169
23    169
dtype: int64


In [ ]:
# CHECK FOR DUPLICATE IN DECEMBER 2
with zipfile.ZipFile('DATA/202512/PassengerOD_31122568.zip') as zf:
    print(zf.namelist())

# Dec 31, 2025 includes 22 hourly records (extended New Year's Eve service to 03:00 on Jan 1), vs. the standard 20-hour operating day.

['PassengerOD_31122568_10-11.xlsx', 'PassengerOD_31122568_11-12.xlsx', 'PassengerOD_31122568_12-13.xlsx', 'PassengerOD_31122568_13-14.xlsx', 'PassengerOD_31122568_14-15.xlsx', 'PassengerOD_31122568_15-16.xlsx', 'PassengerOD_31122568_16-17.xlsx', 'PassengerOD_31122568_17-18.xlsx', 'PassengerOD_31122568_18-19.xlsx', 'PassengerOD_31122568_19-20.xlsx', 'PassengerOD_31122568_20-21.xlsx', 'PassengerOD_31122568_21-22.xlsx', 'PassengerOD_31122568_22-23.xlsx', 'PassengerOD_31122568_23-00.xlsx', 'PassengerOD_01012569_00-01.xlsx', 'PassengerOD_01012569_01-02.xlsx', 'PassengerOD_01012569_02-03.xlsx', 'PassengerOD_31122568_05-06.xlsx', 'PassengerOD_31122568_06-07.xlsx', 'PassengerOD_31122568_07-08.xlsx', 'PassengerOD_31122568_08-09.xlsx', 'PassengerOD_31122568_09-10.xlsx']


___
**END**